# Stage 2: more model changes on SmolVLA (no new reference runs)
Reuses the 400 reference episodes from the go/no-go (dataset **gonogo-v5**) and runs new variants on the same 140
paired test slots, then re-scores every saved run with 5 monitor families x 3 placements, plus 20 random re-splits of calibration.

| PART | Runs | Approx. 2xT4 time |
|---|---|---|
| 1 | shadow replay on repair slots (W4-LM, W8, 5 denoising steps) + **W4 on the VLM language model + expert** (test + repair slots) | ~9 h |
| 2 | **W8 on the expert**, test + repair slots | ~5 h |
| 3 | **5 denoising steps** (runtime-config shift, same weights), test + repair slots | ~4 h |
| 4 | **5 actions per call** (runtime-config shift) + W4 calibration pool (repair slots) | ~7 h |

**Setup (once):** Settings -> Accelerator **GPU T4 x2**, Internet **On**. Add Input: dataset **vla-assure** (latest),
dataset **gonogo-v5**. Then set `PART = 1` in cell 2a, **Save Version -> Save & Run All (Commit)**.
**Then:** `PART = 2`, `3`, `4`, one commit each (no need to attach earlier outputs).
**Send back:** `stage2_partN.zip` from each version's Output tab.

## 1. Setup

In [ ]:
# 1a. GPUs, time guard, work folder; resume from a previous version's output if one is attached as input.
import os, sys, time, json, glob, shutil, pathlib, subprocess
T_START = time.time()
TIME_BUDGET_H = 11.0          # stop cleanly before Kaggle's 12 h limit so the output is saved
def hours_left():
    return TIME_BUDGET_H - (time.time() - T_START) / 3600

print(subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
NGPU = int(subprocess.run("nvidia-smi -L | wc -l", shell=True, capture_output=True, text=True).stdout.strip() or 0)
assert NGPU >= 1, "No GPU: right panel > Accelerator > GPU T4 x2"
print("GPUs:", NGPU, "| kernel Python", sys.version.split()[0])

WORK = pathlib.Path("/kaggle/working/vla-gonogo"); WORK.mkdir(parents=True, exist_ok=True)
# Kaggle mounts inputs at different depths (e.g. /kaggle/input/notebooks/<user>/<slug>/vla-gonogo)
prev = sorted({p for p in glob.glob("/kaggle/input/**/vla-gonogo", recursive=True) if pathlib.Path(p).is_dir()})
for p in prev:
    if pathlib.Path(p).resolve() != WORK.resolve():
        print("Resuming from previous output:", p)
        shutil.copytree(p, WORK, dirs_exist_ok=True, symlinks=False)
n_done = len(list(WORK.rglob("*.npz")))
print("Work folder:", WORK, "|", n_done, "episodes already on disk")
PERSISTENT = True   # /kaggle/working is saved as the version's output

In [ ]:
# 1b. Code: the newest vla-assure copy among the inputs (the one with the stage-2 variants)
import zipfile
CODE = pathlib.Path("/root/vla-assure")

def code_ok(root):
    f = root / "src/assure/adapters/lerobot_adapter.py"
    return f.exists() and "_chunk_method" in f.read_text() and "_require_lerobot_06" in f.read_text()

def newest(root):
    root = pathlib.Path(root)
    f, y = root / "scripts/shadow_check.py", root / "configs/experiment.yaml"
    return (f.exists() and "all_taps" in f.read_text() and y.exists() and "w4_lm" in y.read_text()
            and (root / "scripts/resplit.py").exists())

if CODE.exists() and not (code_ok(CODE) and newest(CODE)):
    shutil.rmtree(CODE)
if not CODE.exists():
    trees = sorted({str(pathlib.Path(p).parents[3]) for p in
                    glob.glob("/kaggle/input/**/src/assure/adapters/lerobot_adapter.py", recursive=True)})
    trees = [t for t in trees if newest(t)]
    if not trees:
        raise FileNotFoundError("No vla-assure copy with the stage-2 variants (w4_lm + resplit.py) under /kaggle/input: "
                                "upload the newest vla-assure.zip as a dataset version and attach it")
    print("Using unpacked dataset", trees[0])
    shutil.copytree(trees[0], CODE, symlinks=False)
assert code_ok(CODE), "This code is outdated: upload the latest vla-assure.zip"
for d in ("rollouts", "results"):
    (WORK / d).mkdir(exist_ok=True)
    tgt = CODE / d
    if tgt.is_symlink():
        tgt.unlink()
    elif tgt.exists():
        shutil.rmtree(tgt)
    tgt.symlink_to(WORK / d)
os.chdir(CODE); print("Code OK at", os.getcwd())

In [ ]:
# 1c. Python 3.12 venv for LeRobot 0.6.1 (needs Python >= 3.12; Kaggle's kernel env is left untouched).
#     egl_probe / hf-egl-probe (via hf-libero -> robomimic) are C++ packages that fail to build with CMake 4;
#     LeRobot's LIBERO env never imports them, so tiny pure-Python stand-ins are installed first.
VENV = pathlib.Path("/root/py312"); PY = str(VENV / "bin/python")
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                       env=dict(os.environ, CMAKE_POLICY_VERSION_MINIMUM="3.5"))
    if r.returncode != 0 or not check:
        print(r.stdout[-1500:], r.stderr[-3000:])
    if check:
        assert r.returncode == 0, f"failed: {cmd}"
    return r
if not pathlib.Path(PY).exists():
    sh(f"{sys.executable} -m pip install -q uv")
    sh(f"{sys.executable} -m uv venv --seed --python 3.12 {VENV}")
stub = pathlib.Path("/root/_stubs")
(stub / "egl_probe/egl_probe").mkdir(parents=True, exist_ok=True); (stub / "hf_egl_probe").mkdir(parents=True, exist_ok=True)
(stub / "egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "egl_probe"\nversion = "1.0.2"\n[tool.setuptools]\npackages = ["egl_probe"]\n')
(stub / "egl_probe/egl_probe/__init__.py").write_text("def get_available_devices():\n    return [0]\n")
(stub / "hf_egl_probe/pyproject.toml").write_text('[build-system]\nrequires = ["setuptools>=61"]\nbuild-backend = "setuptools.build_meta"\n'
    '[project]\nname = "hf-egl-probe"\nversion = "1.0.2"\n[tool.setuptools]\npy-modules = []\n')
sh(f"{PY} -m pip install -q {stub / 'egl_probe'} {stub / 'hf_egl_probe'}")
t = time.time()
sh(f'{PY} -m pip install -q "lerobot[smolvla,libero]==0.6.1" pandas scipy tabulate pyyaml')
print(f"installed in {time.time() - t:.0f} s:",
      sh(f"{PY} -c \"import importlib.metadata as m, torch; print('lerobot', m.version('lerobot'), '| torch', "
         f"torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count())\"").stdout)

In [ ]:
# 1d. Headless rendering. Try NVIDIA EGL first; if MuJoCo cannot render, install Mesa EGL (software) and retry;
#     last resort OSMesa. The chosen backend is used by every later step.
probe = ("import mujoco; m = mujoco.MjModel.from_xml_string('<mujoco><worldbody><light pos=\"0 0 1\"/>"
         "<geom type=\"sphere\" size=\".1\"/></worldbody></mujoco>'); r = mujoco.Renderer(m, 64, 64); "
         "d = mujoco.MjData(m); mujoco.mj_forward(m, d); r.update_scene(d); print('render ok', r.render().shape)")
def try_gl(gl):
    r = subprocess.run([PY, "-c", probe], capture_output=True, text=True,
                       env=dict(os.environ, MUJOCO_GL=gl, PYOPENGL_PLATFORM=gl))
    return r.returncode == 0, (r.stdout + r.stderr)[-400:]
GL = None
for attempt in ("egl", "egl+mesa", "osmesa"):
    if attempt == "egl+mesa":
        sh("apt-get -qq update && apt-get -qq install -y libegl1 libegl-mesa0 libgl1 libgl1-mesa-dri >/dev/null", check=False)
    if attempt == "osmesa":
        sh("apt-get -qq install -y libosmesa6 >/dev/null", check=False)
    ok, log = try_gl(attempt.split("+")[0])
    print(attempt, "->", "OK" if ok else log)
    if ok:
        GL = attempt.split("+")[0]; break
assert GL, "no rendering backend works"
if attempt == "egl+mesa":
    print("NOTE: software EGL (Mesa). Rendering is CPU-bound; expect slower episodes than on Colab.")
os.environ["MUJOCO_GL"] = GL; os.environ["PYOPENGL_PLATFORM"] = GL
os.environ["MPLBACKEND"] = "Agg"   # the kernel's inline backend is not valid inside the venv's matplotlib
# LIBERO's first import asks interactive questions and downloads assets: answer once, non-interactively.
r = subprocess.run([PY, "-c", "import libero.libero as L; print('assets:', L.get_libero_path('assets'))"],
                   input="n\n", capture_output=True, text=True, env=dict(os.environ))
print(r.stdout[-600:], r.stderr[-600:])

Configuration and helpers (same settings as the Colab notebook).

In [ ]:
# 1e. Experiment configuration for this gate (written into the repo copy on this machine only)
import yaml, json
CFG = CODE / "configs"
exp = yaml.safe_load((CFG / "experiment.yaml").read_text())
models = yaml.safe_load((CFG / "models.yaml").read_text())

MODEL, SUITE = "smolvla", "libero_spatial"
R_REF = 40              # reference slots per task (500 slots: 125 train / 100 calib / 100 repair / 175 test)
BUDGET_PCT = 15         # % of the ACTION EXPERT's linear weights kept in full precision (~5 of 32 blocks)
N_ACTION_STEPS = 10     # 10 actions per model call (~5x faster than the checkpoint's 1); same for every variant

exp["quant_scope"] = {"include": ["model.vlm_with_expert.lm_expert.*"],
                      "block_regex": r"lm_expert\.layers\.(\d+)$"}
exp["monitors"] = ["mahalanobis#A", "mahalanobis"]
exp["reference_episodes_per_task"] = R_REF
models["models"][MODEL]["checkpoint"] = "HuggingFaceVLA/smolvla_libero"
models["models"][MODEL]["n_action_steps"] = N_ACTION_STEPS
(CFG / "experiment.yaml").write_text(yaml.safe_dump(exp, sort_keys=False))
(CFG / "models.yaml").write_text(yaml.safe_dump(models, sort_keys=False))
CONDITION, KEY = "nominal", SUITE   # fixed by the go/no-go
VARIANTS = []                         # set per PART in cell 2a
print("taps:", models["models"][MODEL]["tap"])

def run(cmd, tail=3000):
    # Run a pipeline script, stream nothing, print the end of the log, raise on failure.
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    p = subprocess.run([PY] + cmd, capture_output=True, text=True, env=env, input="n\n")
    out = (p.stdout + p.stderr).replace("\r", "\n")
    out = "\n".join(l for l in out.splitlines() if "Running rollout" not in l and "it/s" not in l)
    print(out[-tail:])
    if p.returncode != 0:
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


class OutOfTime(Exception):
    pass

def _clean_partial(since):
    # remove episode files that a killed shard may have left half-written
    import numpy as np
    for f in (CODE / "rollouts").rglob("*.npz"):
        if f.stat().st_mtime >= since - 120:
            try:
                np.load(f)["success"]
            except Exception:
                print("removing partial file", f); f.unlink()

def collect(args, tail=1500):
    # 01_collect.py sharded by task over the GPUs, then one merge pass (skips finished episodes) that writes the
    # complete run.json. Stops cleanly when TIME_BUDGET_H is reached; rerun (next version) to resume.
    if hours_left() < 0.25:
        raise OutOfTime("time budget reached: Save Version now; attach this output and run again to resume")
    args = list(args) + ([] if "--skip_existing" in args else ["--skip_existing"])
    env = dict(os.environ, MUJOCO_GL=GL, PYOPENGL_PLATFORM=GL, TOKENIZERS_PARALLELISM="false")
    if NGPU > 1 and "--tasks" not in args:
        logs = [open(f"/root/shard{g}.log", "w") for g in range(NGPU)]
        t0 = time.time()
        procs = [subprocess.Popen([PY, "scripts/01_collect.py"] + args +
                                  ["--tasks", ",".join(str(t) for t in range(10) if t % NGPU == g)],
                                  # all shards on GPU 0 (the setup that works with robosuite's EGL checks); SmolVLA is small and the
                                  # simulator is CPU-bound, so two processes on one T4 still give ~2x
                                  env=dict(env, CUDA_VISIBLE_DEVICES="0"), stdout=logs[g], stderr=subprocess.STDOUT,
                                  stdin=subprocess.PIPE, text=True) for g in range(NGPU)]
        for p in procs:
            p.stdin.write("n\n"); p.stdin.close()
        # poll: stop all shards at once if one fails or the time budget runs out
        bad = []
        while any(p.poll() is None for p in procs):
            bad = [g for g, p in enumerate(procs) if p.poll() not in (None, 0)]
            late = hours_left() * 3600 < 600
            if bad or late:
                for p in procs:
                    if p.poll() is None:
                        p.kill()
                for p in procs:
                    p.wait()
                _clean_partial(t0)
                if late and not bad:
                    raise OutOfTime("time budget reached during collection: Save Version; attach this output and run again")
                break
            time.sleep(10)
        for f in logs:
            f.close()
        for g, p in sorted(enumerate(procs), key=lambda x: x[0] not in bad):   # the shard that failed first
            if p.returncode != 0:
                print(open(f"/root/shard{g}.log").read()[-3000:])
                raise RuntimeError(f"shard {g} failed")
    run(["scripts/01_collect.py"] + args, tail=tail)   # merge pass (or the single-GPU run)

In [ ]:
# 1f. Smoke test: model loads, hooks fire, camera shift applied, 2 episodes run. Measures seconds per episode.
run(["scripts/01_collect.py", "--model", MODEL, "--variant", "fp16", "--suite", SUITE, "--condition", "nominal",
     "--tasks", "0", "--episodes", "2", "--out", str(WORK / "rollouts_smoke")])
rj = json.loads((WORK / f"rollouts_smoke/{MODEL}/fp16/{SUITE}/run.json").read_text())
sec = rj["seconds"] / max(rj["episodes"], 1)
print(f"\nsuccess {rj['success_rate']:.2f} | taps fired {rj['tap_fired']} | {sec:.0f} s/episode")
assert all(v > 0 for v in rj["tap_fired"].values()), "feature hooks did not fire: check tap paths in models.yaml"
for step, n in (("Stage 0-lite", 100), ("condition pilot", 100), ("reference", 10 * R_REF), ("variants", 4 * 175)):
    print(f"  {step:16s} {n:5d} episodes ~ {n * sec / 3600:5.1f} h")
print(f"With {NGPU} GPUs in parallel divide by {NGPU}. If reference + variants exceeds ~16 h, set "
      "N_ACTION_STEPS = 10 in 1e and rerun.")

## 2. Stage-2 runs

In [ ]:
# 2a. Which part this commit runs (1, 2, 3, then 4)
PART = 1
VARIANTS = {1: ["w4_lm"], 2: ["w8"], 3: ["cfg_denoise5"], 4: ["cfg_steps5", "w4"]}[PART]
# test slots = paired endpoint; repair slots + top-ups (until 44 successes) = the changed policy's OWN calibration
# pool, disjoint from test, so recalibration (19 marginal / 44 PAC) is judged on independent episodes.
ROLES = {"w4": "repair"}   # w4 test slots already exist from the go/no-go; only its calibration pool is new
ref = CODE / f"rollouts/{MODEL}/fp16/{KEY}"
n_ref = len(list(ref.glob("*.npz")))
print("PART", PART, "| variants:", VARIANTS, "| reference episodes on disk:", n_ref)
assert n_ref >= 10 * R_REF, "reference runs missing: attach the gonogo-v5 dataset as input"
if "w4" in VARIANTS:
    n_w4 = len(list((CODE / f"rollouts/{MODEL}/w4/{KEY}").glob("*.npz")))
    print("w4 episodes on disk (go/no-go test slots):", n_w4)
    assert n_w4 >= 140, "w4 test episodes missing: attach the gonogo-v5 dataset (it holds the go/no-go W4 run)"

In [ ]:
# 2b. Shadow replay on identical frames (PART 1 only; ~1.5-2 h): W4-LM, W8 and 5 denoising steps see the fp16 robot's frames.
#     Runs on REPAIR slots, disjoint from the closed-loop test slots, so the prediction is judged out of sample.
if PART == 1:
    collect(["--model", MODEL, "--variant", "fp16", "--shadow", "w4_lm,w8,cfg_denoise5", "--suite", SUITE,
             "--condition", CONDITION, "--episodes", str(R_REF), "--ref_roles", "repair", "--max_per_task", "3",
             "--out", str(WORK / "rollouts_shadow2")])
    for taps in ("vlm_last,expert_last", "expert_last", "vlm_last"):
        run(["scripts/shadow_check.py", "--model", MODEL, "--key", KEY, "--shadow_root", str(WORK / "rollouts_shadow2"),
             "--taps", taps, "--out", str(WORK / f"shadow2_{taps.replace(',', '+')}.json")], tail=1800)

In [ ]:
# 2c. Closed-loop variants on the 140 paired test slots (resumable)
for v in VARIANTS:
    print("=" * 30, v)
    collect(["--model", MODEL, "--variant", v, "--suite", SUITE, "--condition", CONDITION,
             "--episodes", str(R_REF), "--roles", ROLES.get(v, "test,repair"),
             "--target_successes", "44", "--max_extra_rounds", "4"], tail=600)
    rj = json.loads((CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json").read_text())
    print(f"{v}: success {rj['success_rate']:.3f} | sampler {rj.get('sampler')}")

In [ ]:
# 2d. Re-score every saved run with 5 monitor families x 3 placements (CPU part, ~30-60 min)
import pandas as pd, numpy as np
OUT = {}
for name, taps in (("both_layers", "vlm_last,expert_last"), ("expert_only", "expert_last"), ("vlm_only", "vlm_last")):
    print("scoring", name, flush=True)
    run(["scripts/03_evaluate.py", "--model", MODEL, "--key", KEY,
         "--monitors", "mahalanobis,rnd,safe_mlp,acc,ext_mahalanobis", "--taps", taps], tail=400)
    OUT[name] = WORK / f"trials_part{PART}_{name}.csv"
    shutil.copy(CODE / f"results/eval_trials/{MODEL}__{KEY}.csv", OUT[name])
for name, taps in (("both_layers", "vlm_last,expert_last"), ("expert_only", "expert_last"), ("vlm_only", "vlm_last")):
    run(["scripts/resplit.py", "--model", MODEL, "--key", KEY, "--taps", taps, "--variants", ",".join(VARIANTS),
         "--splits", "20", "--out", str(WORK / f"resplit_part{PART}_{name}.json")], tail=3000)
rows = []
for name, path in OUT.items():
    t = pd.read_csv(path); t = t[(t.alpha.round(3) == 0.10) & (t.ref_success >= 0) & t.variant.isin(VARIANTS)]
    for (mon, v), d in t.groupby(["monitor", "variant"]):
        js = d[(d.ref_success == 1) & (d.var_success == 1)]
        nf = d[(d.ref_success == 1) & (d.var_success == 0)]
        rows.append(dict(features=name, monitor=mon, variant=v, success_fp16=d.ref_success.mean(),
                         success_var=d.var_success.mean(), joint_succ=len(js), FPR_fp16=js.ref_alarm.mean(),
                         FPR_var=js.var_alarm.mean(), n_new_failures=len(nf),
                         detect_NEW_failures=nf.var_alarm.mean() if len(nf) else np.nan))
res = pd.DataFrame(rows).round(3)
pd.set_option("display.width", 250); print(res.to_string(index=False))
res.to_csv(WORK / f"groups_part{PART}.csv", index=False)
with zipfile.ZipFile(f"/kaggle/working/stage2_part{PART}.zip", "w") as z:
    for p in list(OUT.values()) + [WORK / f"groups_part{PART}.csv"] + list(WORK.glob("shadow2_*.json")) + list(WORK.glob(f"resplit_part{PART}_*.json")):
        z.write(p, p.name)
    for v in VARIANTS:
        z.write(CODE / f"rollouts/{MODEL}/{v}/{KEY}/run.json", f"run_{v}.json")
print(f"Send me /kaggle/working/stage2_part{PART}.zip (Output tab)")